# ch12 类别不平衡（练习版）

> 按提示补全 `____`，跑通所有 assert。数据、划分、预处理与 `skf` 已就绪。
>
> 注意：挖空块内每条语句保持**单行**；循环/分支写在挖空块之外。

In [ ]:
from pathlib import Path

import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")

In [ ]:
# 数据与划分口径沿用 ch05：分层 8:2、median 填充、独热、标准化
dev = pd.read_csv(DATA / "defect_ml.csv")
NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"
y_all = dev[TARGET]

tr_idx, te_idx = train_test_split(np.arange(len(dev)), test_size=0.2,
                                  random_state=42, stratify=y_all)
dev_tr, dev_te = dev.iloc[tr_idx], dev.iloc[te_idx]
ys_tr, ys_te = dev_tr[TARGET], dev_te[TARGET]

imp = SimpleImputer(strategy="median").fit(dev_tr[NUM])
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(dev_tr[CAT])
feat = NUM + [f"{c}_{v}" for c, vals in zip(CAT, oh.categories_) for v in vals]
Xtr = np.hstack([imp.transform(dev_tr[NUM]), oh.transform(dev_tr[CAT])])
Xte = np.hstack([imp.transform(dev_te[NUM]), oh.transform(dev_te[CAT])])
ss = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("sklearn", sklearn.__version__, "| imblearn",
      __import__("imblearn").__version__)
print("训练正例:", int(ys_tr.sum()), "/", len(ys_tr),
      "| 测试正例:", int(ys_te.sum()), "/", len(ys_te))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def report_cm(tag, yt, yp):
    """一行报 acc/P/R/f1 与混淆矩阵，返回 (acc, recall, f1)。"""
    acc = round(float(accuracy_score(yt, yp)), 4)
    rec = round(float(recall_score(yt, yp)), 4)
    f1v = round(float(f1_score(yt, yp, zero_division=0)), 4)
    print(f"{tag}: acc={acc} P={round(float(precision_score(yt, yp, zero_division=0)), 4)} "
          f"R={rec} f1={f1v} cm={confusion_matrix(yt, yp).tolist()}")
    return acc, rec, f1v


print("脚手架就绪：report_cm")

## 任务 1：基线体检

默认阈值下 acc 与 recall 的巨大反差。

In [ ]:
# TODO(1)：基线体检：默认阈值下 0.7833 的准确率是怎么来的
# 提示：base = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr_s, ys_tr)；report_cm("基线", ys_te, base.predict(Xte_s))
base = ____
base_acc, base_rec, base_f1 = ____
print(f"全判负也能拿 {round(float(1 - ys_te.mean()), 4)}——基线只比『躺平』多抓住 {int(confusion_matrix(ys_te, base.predict(Xte_s))[1, 1])} 个正例")

assert base_acc == 0.7833
assert base_rec == 0.1923, "召回只有 0.19：26 个危急漏掉 21 个"
assert base_f1 == 0.2778, "f1 被 21% 正例拉到难看——这才是真实成绩单"

## 任务 2：class_weight=balanced

FN 21 → 5 的零成本方案。

In [ ]:
# TODO(2)：class_weight=balanced：按类频率反加权，FN 直接 21 → 5
# 提示：LogisticRegression(..., class_weight="balanced")
cw = ____
cw_acc, cw_rec, cw_f1 = ____
print(f"acc {base_acc} → {cw_acc}，recall {base_rec} → {cw_rec}：用 8 个点准确率换 6 倍召回")

assert cw_acc == 0.7 and cw_rec == 0.8077, "FN 21 → 5"
assert cw_f1 == 0.5385, "f1 几乎翻倍（0.2778 → 0.5385）"
assert cw_acc < base_acc and cw_rec > base_rec, "训练时调的是损失，测试时看的指标必须换"

## 任务 3：自定义权重

`{0: 1, 1: 4}` 比全自动更激进。

In [ ]:
# TODO(3)：自定义权重：{0:1, 1:4} 比全自动更进一步
# 提示：class_weight 传字典 {0: 1, 1: 4}
cw4 = ____
cw4_acc, cw4_rec, cw4_f1 = ____
print("balanced 的隐式权重:", {0: round(float(len(ys_tr) / (2 * (len(ys_tr) - ys_tr.sum()))), 4),
                            1: round(float(len(ys_tr) / (2 * int(ys_tr.sum()))), 4)})

assert cw4_acc == 0.7 and cw4_rec == 0.8462 and cw4_f1 == 0.55, \
    "手动 4 倍比 balanced 的自动 ~2.35 倍更激进，f1 再挤一点"
assert cw4_rec > cw_rec, "权重越大 → 正例损失越贵 → 模型越敢报正例"

## 任务 4：阈值移动

不重训只动判决线，并发现与 balanced 的等价性。

In [ ]:
# TODO(4)：阈值移动：不重训模型，只动判决线
# 提示：p_te = base.predict_proba(Xte_s)[:, 1]；yp = (p_te >= t).astype(int)
p_te = ____
thr_acc, thr_rec, thr_f1 = ____
t02_cm = ____
print("阈值 0.2 的 cm:", t02_cm.tolist())
print("对照 balanced 的 cm:", confusion_matrix(ys_te, cw.predict(Xte_s)).tolist())

assert thr_acc == 0.7583 and thr_rec == 0.6538, "阈值 0.5 → 0.3：召回 0.19 → 0.65"
assert thr_f1 == 0.5397, "f1 0.5397 甚至超过 balanced 的 0.5385"
assert t02_cm.tolist() == confusion_matrix(ys_te, cw.predict(Xte_s)).tolist(), \
    "阈值 0.2 的判决和 balanced 完全一致——class_weight 就是隐式的阈值移动"

## 任务 5：SMOTE 正确用法

只对训练集插值，看样本量与成绩变化。

In [ ]:
# TODO(5)：SMOTE 正确用法：只对训练集插值造样本
# 提示：sm = SMOTE(random_state=42)；Xtr_sm, ys_tr_sm = sm.fit_resample(Xtr_s, ys_tr)
sm = ____
Xtr_sm, ys_tr_sm = ____
lr_sm = ____
sm_acc, sm_rec, sm_f1 = ____
print("SMOTE 前:", Xtr_s.shape, "正例", int(ys_tr.sum()))
print("SMOTE 后:", Xtr_sm.shape, "正例", int(ys_tr_sm.sum()))

assert Xtr_sm.shape == (756, 16) and int(ys_tr_sm.sum()) == 378, \
    "auto 策略补到 1:1：102 → 378 正例（新增 276 个合成点）"
assert sm_rec == 0.7692 and sm_acc == 0.6917, "SMOTE 效果与 balanced 一个档次，但多了造样本的随机性"

## 任务 6：作弊示范

在测试集上 SMOTE 会发生什么（只围观，不模仿）。

In [ ]:
# TODO(6)：经典作弊：在测试集上 SMOTE（错误示范，看数字怎么失真）
# 提示：对 Xte_s 也 fit_resample 就污染了；结果只在 print 里围观，不许写进报告
Xte_sm, ys_te_sm = ____
bad_acc = ____
print("作弊后的『acc』:", bad_acc, "（测试样本被人工补成 1:1，此数字作废）")
print("正确姿势: SMOTE 只碰训练集，测试集永远保持原始分布")

assert len(ys_te_sm) != len(ys_te), "测试集 120 → 188，已经不是同一道题了"
assert bad_acc != round(float(accuracy_score(ys_te, lr_sm.predict(Xte_s))), 4), \
    "污染前后数字对不上——一旦混入提交结果就是事故"

## 任务 7：imblearn Pipeline

CV 里防泄漏的正确姿势。

In [ ]:
# TODO(7)：imblearn Pipeline：CV 里防 SMOTE 泄漏的正确姿势
# 提示：ImbPipeline([("smote", SMOTE(random_state=42)), ("clf", LogisticRegression(max_iter=1000, random_state=42))])；cross_val_score(..., scoring="recall", cv=skf)
imb_pipe = ____
sm_sc = ____
sm_cv = ____
plain_sc = ____
plain_cv = ____
print("SMOTE 管道 CV recall 各折:", np.round(sm_sc, 4).tolist(), "| mean", sm_cv)
print("普通 LR  CV recall 各折:", np.round(plain_sc, 4).tolist(), "| mean", plain_cv)

assert sm_cv == 0.6067 and plain_cv == 0.3533, "SMOTE 在 CV 内平均把召回拉高 25 个点"
assert sm_cv > plain_cv, "imblearn 的 Pipeline 会在每折训练段内部重采样：验证折保持原味"

## 任务 8：三方案对表

balanced / SMOTE / 阈值移动，同一测试集见真章。

In [ ]:
# TODO(8)：三方案对表：不平衡数据选型一眼定
# 提示：三个已拟合的模型 + 一条手工阈值，全部过 report_cm
acc_b, rec_b, f1_b = ____
acc_s, rec_s, f1_s = ____
acc_t, rec_t, f1_t = ____
print("选型口径: 漏报代价大 → 保 recall（balanced）；只要排序 → 原模型+移动阈值；"
      "要造样本 → SMOTE（且只在训练集）")

assert max(f1_b, f1_s, f1_t) == f1_t == 0.5397, "本数据上『不动模型只动阈值』f1 最高"
assert acc_t > acc_b > acc_s, "阈值移动连 acc 都保得住：0.7583 > 0.70 > 0.6917"

## 完成标准

所有 assert 通过 = 任务完成。自查：balanced 的隐式权重公式、
阈值 0.2 为什么和 balanced 同判决、SMOTE 为什么只碰训练集、
CV 防泄漏的正确姿势。